# Application Data – Feature Engineering

Cleans and enriches the `application_train` dataset for the Home Credit default prediction pipeline.

**Pipeline overview**
1. Load raw data
2. Clean invalid sentinel values (`365243`, `XNA`)
3. Derive financial ratios & demographic features
4. Document & contact flags
5. Phone-change recency
6. Housing indices
7. Binary property & amenity flags
8. Quality checks & save

## 1. Imports & Configuration

In [ ]:
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = r"a:\Dominic\My Projects\Home Credit"
os.chdir(PROJECT_ROOT)

PATH_APP_TRAIN = "data/Internal/application_train.csv"
PATH_OUTPUT    = os.path.join(
    PROJECT_ROOT, "src", "intermediate",
    "application_train_data_processed.parquet"
)

## 2. Load Data

In [ ]:
application_train = pd.read_csv(PATH_APP_TRAIN)
application_train = application_train.astype({'SK_ID_CURR': 'string'})

print(f"Shape: {application_train.shape}")

## 3. Clean Invalid Sentinel Values

### 3a. Numeric sentinel `365243`

`365243` is used as a placeholder for missing values in several `DAYS_*` columns
(it represents an implausibly large positive number of days). We scan *all* numeric
columns for this value and replace every occurrence with `NaN`, rather than targeting
only `DAYS_EMPLOYED` as the original did.

> **Fix:** the original only cleaned `DAYS_EMPLOYED`. The scan below catches any
> other column that may also carry this sentinel.

In [ ]:
SENTINEL = 365243
numeric_cols = application_train.select_dtypes(include='number').columns
sentinel_cols = [
    col for col in numeric_cols
    if (application_train[col] == SENTINEL).any()
]
print(f"Columns containing sentinel {SENTINEL}: {sentinel_cols}")

for col in sentinel_cols:
    application_train.loc[application_train[col] == SENTINEL, col] = np.nan

print("Sentinel values replaced with NaN.")

### 3b. String sentinel `'XNA'`

`XNA` is used in categorical columns to denote unknown values. Both
`ORGANIZATION_TYPE` and `CODE_GENDER` are known to carry this value.

> **Fix:** the original only cleaned `ORGANIZATION_TYPE`. `CODE_GENDER = 'XNA'`
> was left in, which would cause issues at any encoding step.

In [ ]:
object_cols = application_train.select_dtypes(include='object').columns
xna_cols = [
    col for col in object_cols
    if (application_train[col] == 'XNA').any()
]
print(f"Columns containing 'XNA': {xna_cols}")

for col in xna_cols:
    application_train.loc[application_train[col] == 'XNA', col] = np.nan

print("XNA values replaced with NaN.")

## 4. Financial Ratios & Demographic Features

- **`GOODSPRICE_TO_AMT_CREDIT`** — how much of the credit covers goods vs. total credit.
- **`AMT_ANNUITY_TO_CREDIT_RATIO`** — annual repayment burden relative to loan size.
- **`AMT_ANNUITY_TO_INCOME_RATIO`** — repayment burden relative to income (debt-service ratio).
- **`INCOME_PER_FAMILY`** — per-capita income within the household.
- **`AGE`** — derived from `DAYS_BIRTH` (always negative → positive years).
- **`EMPLOYED_YEARS`** — derived from `DAYS_EMPLOYED`; safe to compute after sentinel cleaning.
- **`EXT_SOURCES_MEAN/MIN/MAX`** — summary statistics across the three external credit scores.

> `DAYS_BIRTH / -365` uses 365 days per year (≈ 365.25 with leap years) — the small
> rounding difference is negligible for a feature used as an age proxy.

In [ ]:
application_train['GOODSPRICE_TO_AMT_CREDIT'] = np.where(
    application_train['AMT_CREDIT'] > 0,
    application_train['AMT_GOODS_PRICE'] / application_train['AMT_CREDIT'],
    np.nan,
)
application_train['AMT_ANNUITY_TO_CREDIT_RATIO'] = np.where(
    application_train['AMT_CREDIT'] > 0,
    application_train['AMT_ANNUITY'] / application_train['AMT_CREDIT'],
    np.nan,
)
application_train['AMT_ANNUITY_TO_INCOME_RATIO'] = np.where(
    application_train['AMT_INCOME_TOTAL'] > 0,
    application_train['AMT_ANNUITY'] / application_train['AMT_INCOME_TOTAL'],
    np.nan,
)
application_train['INCOME_PER_FAMILY'] = np.where(
    application_train['CNT_FAM_MEMBERS'] > 0,
    application_train['AMT_INCOME_TOTAL'] / application_train['CNT_FAM_MEMBERS'],
    np.nan,
)

application_train['AGE']            = application_train['DAYS_BIRTH']    / -365
application_train['EMPLOYED_YEARS'] = application_train['DAYS_EMPLOYED'] / -365

ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
application_train['EXT_SOURCES_MEAN'] = application_train[ext_cols].mean(axis=1)
application_train['EXT_SOURCES_MIN']  = application_train[ext_cols].min(axis=1)
application_train['EXT_SOURCES_MAX']  = application_train[ext_cols].max(axis=1)

print("Financial & demographic features added.")

## 5. Document & Contact Flags

- **`TOTAL_DOCUMENTS_SUBMITTED`** — count of submitted supporting documents (sum of all `FLAG_DOCUMENT_*` columns).
- **`CONTACT_REACHABILITY_SCORE`** — sum of 5 binary contact flags; ranges 0–5.

In [ ]:
doc_cols = [col for col in application_train.columns if col.startswith('FLAG_DOCUMENT_')]
application_train['TOTAL_DOCUMENTS_SUBMITTED'] = application_train[doc_cols].sum(axis=1)

contact_flags = ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL']
application_train['CONTACT_REACHABILITY_SCORE'] = application_train[contact_flags].sum(axis=1)

print(f"Document columns summed: {len(doc_cols)}")
print(f"Contact score range: [{application_train['CONTACT_REACHABILITY_SCORE'].min()}, "
      f"{application_train['CONTACT_REACHABILITY_SCORE'].max()}]")

## 6. Phone-Change Recency

`DAYS_LAST_PHONE_CHANGE` is a negative value (days before application date).

- **`YEARS_SINCE_PHONE_CHANGE`** — positive years since the last phone change.
- **`RECENT_PHONE_CHANGE_30D`** — `1` if the phone was changed within the last 30 days
  (i.e. `DAYS_LAST_PHONE_CHANGE >= -30`); `NaN` rows map to `0` via `astype(int)`.

In [ ]:
application_train['YEARS_SINCE_PHONE_CHANGE'] = (
    application_train['DAYS_LAST_PHONE_CHANGE'] / -365
)
# DAYS_LAST_PHONE_CHANGE >= -30 → change happened within last 30 days → flag = 1
application_train['RECENT_PHONE_CHANGE_30D'] = (
    application_train['DAYS_LAST_PHONE_CHANGE'] >= -30
).astype(int)

print("Phone-change features added.")

## 7. Housing Area Indices

The Home Credit dataset provides three normalised versions of each building attribute:
`_AVG` (average across flats), `_MEDI` (median), and `_MODE` (most common value).
We compute a composite index for each version by taking the mean across all 14 attributes.
`pandas` `.mean(axis=1)` automatically excludes `NaN` columns, so customers with partial
building data still receive a score. Customers with **no** building data at all get `NaN`.

> **Fix:** the first index was named `TOTAL_HOUSING_AREA_INDEX` (no suffix). Renamed to
> `TOTAL_HOUSING_AREA_INDEX_AVG` for consistency with the `_MEDIAN` and `_MODE` variants.

In [ ]:
housing_avg_cols = [
    'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG',
    'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG',
    'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG',
    'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG',
]
housing_medi_cols = [
    'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI',
    'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI',
    'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI',
    'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI',
]
housing_mode_cols = [
    'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE',
    'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE',
    'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE',
    'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE',
]

# FIX: renamed from TOTAL_HOUSING_AREA_INDEX → TOTAL_HOUSING_AREA_INDEX_AVG
application_train['TOTAL_HOUSING_AREA_INDEX_AVG']    = application_train[housing_avg_cols].mean(axis=1)
application_train['TOTAL_HOUSING_AREA_INDEX_MEDIAN'] = application_train[housing_medi_cols].mean(axis=1)
application_train['TOTAL_HOUSING_AREA_INDEX_MODE']   = application_train[housing_mode_cols].mean(axis=1)

print("Housing indices added.")

## 8. Binary Property & Amenity Flags

- **`FLAG_OWN_CAR` / `FLAG_OWN_REALTY`** — mapped from `Y/N` strings to `1/0`.
  `NaN` (unknown) is treated as `0` (does not own). This is an explicit modelling
  assumption — unknown ownership is conservatively coded as absence of the asset.

- **`IS_EMERGENCY_STATE`** — `1` if the building has a recorded emergency condition, else `0`.
  Missing values are treated as `0` (no known issue). Buildings without data are assumed safe.

- **`HAS_ELEVATOR`** — `1` if `ELEVATORS_AVG > 0`. Missing `ELEVATORS_AVG` → `0`
  (no elevator data assumed to mean no elevator).

- **`IS_HIGH_RISE`** — `1` if `FLOORSMAX_AVG > 0.2` (normalised score above the 20th-percentile
  threshold used in the original dataset). Missing values → `0`.

- **`PREMIUM_AMENITIES_COUNT`** — sum of the three binary flags above (range 0–3).

In [ ]:
# Y/N → 1/0; NaN (unknown ownership) conservatively treated as 0
application_train['FLAG_OWN_CAR']    = application_train['FLAG_OWN_CAR'].map({'Y': 1, 'N': 0}).fillna(0)
application_train['FLAG_OWN_REALTY'] = application_train['FLAG_OWN_REALTY'].map({'Y': 1, 'N': 0}).fillna(0)

# Emergency state: NaN → 0 (building assumed safe when data is absent)
application_train['IS_EMERGENCY_STATE'] = (
    application_train['EMERGENCYSTATE_MODE'].map({'Yes': 1, 'No': 0}).fillna(0)
)

# Elevator presence: NaN ELEVATORS_AVG evaluates to False → 0
application_train['HAS_ELEVATOR'] = (
    application_train['ELEVATORS_AVG'] > 0
).astype(int)

# High-rise: normalised FLOORSMAX_AVG > 0.2 threshold
application_train['IS_HIGH_RISE'] = (
    application_train['FLOORSMAX_AVG'] > 0.2
).astype(int)

amenity_cols = ['IS_EMERGENCY_STATE', 'HAS_ELEVATOR', 'IS_HIGH_RISE']
application_train['PREMIUM_AMENITIES_COUNT'] = application_train[amenity_cols].sum(axis=1)

print("Property & amenity flags added.")

## 9. Quality Checks

In [ ]:
# Confirm sentinel 365243 is fully gone
remaining_sentinel = [
    col for col in application_train.select_dtypes(include='number').columns
    if (application_train[col] == 365243).any()
]
assert not remaining_sentinel, f"Sentinel still present in: {remaining_sentinel}"
print("No sentinel 365243 values remaining.")

# Confirm XNA is fully gone
remaining_xna = [
    col for col in application_train.select_dtypes(include='object').columns
    if (application_train[col] == 'XNA').any()
]
assert not remaining_xna, f"XNA still present in: {remaining_xna}"
print("No XNA values remaining.")

# Null rates for new features
new_features = [
    'GOODSPRICE_TO_AMT_CREDIT', 'AMT_ANNUITY_TO_CREDIT_RATIO',
    'AMT_ANNUITY_TO_INCOME_RATIO', 'INCOME_PER_FAMILY',
    'AGE', 'EMPLOYED_YEARS',
    'EXT_SOURCES_MEAN', 'EXT_SOURCES_MIN', 'EXT_SOURCES_MAX',
    'TOTAL_DOCUMENTS_SUBMITTED', 'CONTACT_REACHABILITY_SCORE',
    'YEARS_SINCE_PHONE_CHANGE', 'RECENT_PHONE_CHANGE_30D',
    'TOTAL_HOUSING_AREA_INDEX_AVG', 'TOTAL_HOUSING_AREA_INDEX_MEDIAN',
    'TOTAL_HOUSING_AREA_INDEX_MODE',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'IS_EMERGENCY_STATE', 'HAS_ELEVATOR', 'IS_HIGH_RISE',
    'PREMIUM_AMENITIES_COUNT',
]
null_rates = application_train[new_features].isna().mean().sort_values(ascending=False)
print("\nNull rates for engineered features:")
print(null_rates[null_rates > 0].to_string())

print(f"\nFinal shape: {application_train.shape}")

## 10. Save Output

In [ ]:
application_train.to_parquet(PATH_OUTPUT, index=False)
print(f"Saved → {PATH_OUTPUT}")